# RAGNav quickstart (offline hybrid retrieval)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/irfanalidv/RAGNav/blob/main/cookbook/ragnav_quickstart.ipynb)

Hybrid **BM25 + sentence-transformers** — no API key. Builds a small index over SQuAD passages and shows **confidence** and **QueryFallback** (retry with a rephrased query when the first pass is uncertain).

In [1]:
%pip install -q "ragnav[embeddings]" "datasets>=2.14.0"


[notice] A new release of pip is available: 25.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
from datasets import load_dataset

from ragnav import RAGNavIndex, RAGNavRetriever
from ragnav.ingest.markdown import ingest_markdown_string

# 50 unique passages from SQuAD validation (order-stable dedupe)
ds = load_dataset("rajpurkar/squad", split="validation[:200]")
seen: set[str] = set()
contexts: list[str] = []
for row in ds:
    c = row["context"]
    if c in seen:
        continue
    seen.add(c)
    contexts.append(c)
    if len(contexts) >= 50:
        break

documents = []
blocks: list = []
for i, ctx in enumerate(contexts):
    doc, doc_blocks = ingest_markdown_string(ctx, name=f"passage_{i}.md")
    documents.append(doc)
    blocks.extend(doc_blocks)

index = RAGNavIndex.build(
    documents=documents,
    blocks=blocks,
    use_sentence_transformers=True,
    vector_model="all-MiniLM-L6-v2",
    embed_batch_size=32,
)
print(f"Index ready: {len(index.blocks_by_id)} blocks from {len(documents)} documents")

/Users/irfanalidv/.pyenv/versions/3.12.1/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(
/Users/irfanalidv/.pyenv/versions/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

Index ready: 10 blocks from 10 documents


In [3]:
retriever = RAGNavRetriever(index=index)

questions = [
    "What is the capital of France?",
    "Who invented the telephone?",
    "When did World War II end?",
]

for q in questions:
    result = retriever.retrieve(
        q,
        top_k=3,
        expand_structure=False,
        expand_graph=False,
    )
    top = result.blocks[0].text[:160].replace("\n", " ") if result.blocks else "(no result)"
    print(f"Q: {q}")
    print(f"Top snippet: {top}...")
    print(f"Confidence: {result.confidence.value}")
    print()

Q: What is the capital of France?
Top snippet: On May 21, 2013, NFL owners at their spring meetings in Boston voted and awarded the game to Levi's Stadium. The $1.2 billion stadium opened in 2014. It is the ...
Confidence: high

Q: Who invented the telephone?
Top snippet: CBS broadcast Super Bowl 50 in the U.S., and charged an average of $5 million for a 30-second commercial during the game. The Super Bowl 50 halftime show was he...
Confidence: medium

Q: When did World War II end?
Top snippet: Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conferenc...
Confidence: medium



## QueryFallback — automatic retry on low / medium confidence

`FakeLLMClient` echoes prompts, so it is a poor query rewriter. For this demo we use a tiny stub that returns **one** alternative line when the fallback asks for variations. In production, plug in any `LLMClient` (e.g. Mistral with `pip install ragnav[mistral]`).

In [4]:
from ragnav.retrieval import QueryFallback


class DemoVariationLLM:
    """Colab-only: return a single rephrasing line for the variation prompt."""

    def chat(self, *, messages, model=None, temperature=0.7):
        return "When was the Great Wall of China built?\n"


fallback = QueryFallback(retriever=retriever, llm=DemoVariationLLM())

# Deliberately vague wording so the first pass is often weaker than the rephrase
out = fallback.retrieve(
    "Great Wall construction date",
    top_k=5,
    expand_structure=False,
    expand_graph=False,
)
print("queries_tried:", out.queries_tried)
print("improved:", out.improved)
print("final confidence:", out.final_result.confidence.value)
if out.final_result.blocks:
    print("top block:", out.final_result.blocks[0].text[:200].replace("\n", " "))

queries_tried: ['Great Wall construction date', 'When was the Great Wall of China built?', 'When was the Great Wall of China built?']
improved: False
final confidence: medium
top block: Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated
